# GAVN fixed-slim: the trimmed geometry arm (3.46M trained params)

One arm of the two-model endgame decided 2026-09-09: replicate the completed 5.30M `gavn-5m-geometry` arm with the unused dynamic attention-bias module physically removed (3,461,377 parameters, bitwise-identical forward to the trained arm), then improve it further one change at a time. CC-GAVN is the other model in the pair and is handled by `07_kaggle_train_ccgavn.ipynb`.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    for _i in range(4):
        _r = subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)])
        if _r.returncode == 0:
            break
        print(f'git clone retry {_i+1}/4 (GitHub rate limit flaps on Kaggle IPs)')
        time.sleep(20 * (_i + 1))
    else:
        raise RuntimeError('git clone failed after retries')
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
SL_REPO = Path('/kaggle/working/searchless_chess')
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
HF_SHARDS = 'chessbench-full-build'  # 8 shards on HF, 5GB peak, no 25GB assemble
assert (REPO / 'scripts/train_gavn.py').exists(), 'clone failed'
for _i in range(18):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
        break
    except Exception as exc:
        print(f'HF secret retry in 10s: {exc}')
        time.sleep(10)
if not os.environ.get('HF_WRITE_TOKEN'):
    import glob as _glob
    for _p in sorted(_glob.glob('/kaggle/input/*/hf_token.txt')):
        os.environ['HF_WRITE_TOKEN'] = Path(_p).read_text().strip()
        print('HF_WRITE_TOKEN loaded from dataset attachment:', _p)
        break
if not os.environ.get('HF_WRITE_TOKEN'):
    print('WARNING: no HF token available; checkpoints will be local only')
import subprocess as _sp
for _i in range(3):
    _r = _sp.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'python-chess'])
    if _r.returncode == 0:
        break
    time.sleep(10)
os.chdir(REPO)
print('setup complete')

In [ ]:
# Mandatory smoke gate.  Besides the persistence/training smoke, assert the
# fixed-slim architecture: the dynamic projection must be physically absent and
# the parameter count must match the verified slim geometry shell exactly.
import torch, json as _json
import sys as _sys
_sys.path.insert(0, str(REPO))
from scripts.train_gavn import GAVN, action_tables, relation_types
_src, _dst, _promo, _u = action_tables(SL_REPO)
_rel = relation_types(legacy=True)  # legacy-v1: replicates the 5.30M geometry arm
_full = GAVN(torch, 224, 8, 8, _src, _dst, _promo, _rel, bias_mode='fixed')
_slim = GAVN(torch, 224, 8, 8, _src, _dst, _promo, _rel, bias_mode='fixed-slim')
_n_full = sum(p.numel() for p in _full.parameters())
_n_slim = sum(p.numel() for p in _slim.parameters())
assert _n_full == 5_304_577, f'unexpected full count {_n_full}'
assert _n_slim == 3_461_377, f'unexpected slim count {_n_slim}'
assert not any('dynamic' in k for k in _slim.state_dict()), 'slim must have no dynamic weights'
print(f'fixed-slim architecture verified: {_n_full:,} -> {_n_slim:,} params')
smoke = Path('/kaggle/working/gavn-slim-smoke')
cmd = [sys.executable, 'scripts/train_gavn.py', '--hf-shards', HF_SHARDS, '--outdir', str(smoke), '--sl-repo', str(SL_REPO), '--dim', '96', '--layers', '2', '--heads', '4', '--batch', '32', '--steps', '20', '--max-records', '4096', '--ckpt-every', '20', '--bias-mode', 'fixed-slim', '--relation-schema', 'legacy-v1', '--w-q', '0.5', '--hf-run', 'smoke-gavn-slim-disposable']
subprocess.run(cmd, check=True)
print('GAVN fixed-slim smoke test passed')

In [ ]:
# Production: the trimmed geometry arm.  Identical to the completed
# gavn-5m-geometry arm (dim 224, 8 layers, fixed bias, legacy-v1 relations,
# w-q 0.5, seed 0, 160k steps) except the dead dynamic branch -- which the
# fixed forward never read, verified bitwise-identical on trained weights --
# is no longer allocated: 3,461,377 trained parameters instead of 5,304,577.
RUN_ID = 'account3-gavn-5m-geometry-slim'
DIM, LAYERS, HEADS = 224, 8, 8
SEED, STEPS = 0, 160000
BIAS_MODE = 'fixed-slim'
RESUME = True
OUT = Path('/kaggle/working') / RUN_ID
cmd = [sys.executable, 'scripts/train_gavn.py', '--hf-shards', HF_SHARDS, '--outdir', str(OUT), '--sl-repo', str(SL_REPO), '--dim', str(DIM), '--layers', str(LAYERS), '--heads', str(HEADS), '--batch', '2048', '--steps', str(STEPS), '--lr', '0.0005', '--warmup', '2000', '--temperature', '1.0', '--bias-mode', BIAS_MODE, '--relation-schema', 'legacy-v1', '--w-dist', '1.0', '--w-q', '0.5', '--w-ce', '0.25', '--seed', str(SEED), '--ckpt-every', '5000', '--hf-repo', 'vedangfake/chess-slm-benchmark', '--hf-run', RUN_ID, '--hf-upload-every', '1800']
if RESUME:
    cmd.append('--resume-from-hf')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Evaluation protocol

Replication gate first: after checkpoint selection by held-out development
loss, evaluate with `notebooks/03_kaggle_eval_frontier.ipynb` (RUN_ID
`account3-gavn-5m-geometry-slim`, `--score auto`) on the frozen protocol. The
replication claim is that the 3.46M slim arm matches the completed 5.30M
geometry arm (84.72% MATE / 43.88% puzzles). Only after that gate may
improvement variants (e.g. the corrected v2 relation schema) be trained on
top, each changing one thing at a time. Never select on the frozen sets.